In [0]:
# ============================================================
# Project: IoT Data Pipeline using Medallion Architecture
# ============================================================

import random
from datetime import datetime
from pyspark.sql.functions import avg, when


# Step 1: Generate sample data
data = []

for i in range(1000):
    data.append((
        f"sensor_{i%5}",
        random.randint(20, 45),
        random.randint(30, 90),
        datetime.now()
    ))

columns = ["device_id", "temperature", "humidity", "timestamp"]

df = spark.createDataFrame(data, columns)


# Bronze Layer
df.write.format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("bronze_iot")

# Silver Layer
df = spark.read.table("bronze_iot")

df = df.dropDuplicates()

df = df.fillna({
    "temperature": 25,
    "humidity": 50
})

df = df.filter((df.temperature > 0) & (df.humidity > 0))

df.write.format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("silver_iot")


# Gold Layer
df = spark.read.table("silver_iot")

df_avg = df.groupBy("device_id").agg(
    avg("temperature").alias("avg_temperature")
)

alerts = df.filter(df.temperature > 40)

df_anomaly = df.withColumn(
    "anomaly",
    when(df.temperature > 42, 1).otherwise(0)
)

df_avg.write.format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("gold_avg")

alerts.write.format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("gold_alerts")

df_anomaly.write.format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("gold_anomaly")


In [0]:
spark.sql("SELECT * FROM bronze_iot").show() #bronze testing


+---------+-----------+--------+--------------------+
|device_id|temperature|humidity|           timestamp|
+---------+-----------+--------+--------------------+
| sensor_0|         28|      54|2026-04-08 10:44:...|
| sensor_1|         39|      45|2026-04-08 10:44:...|
| sensor_2|         41|      75|2026-04-08 10:44:...|
| sensor_3|         31|      51|2026-04-08 10:44:...|
| sensor_4|         35|      45|2026-04-08 10:44:...|
| sensor_0|         44|      52|2026-04-08 10:44:...|
| sensor_1|         40|      34|2026-04-08 10:44:...|
| sensor_2|         23|      58|2026-04-08 10:44:...|
| sensor_3|         45|      68|2026-04-08 10:44:...|
| sensor_4|         30|      66|2026-04-08 10:44:...|
| sensor_0|         21|      56|2026-04-08 10:44:...|
| sensor_1|         32|      83|2026-04-08 10:44:...|
| sensor_2|         27|      54|2026-04-08 10:44:...|
| sensor_3|         28|      41|2026-04-08 10:44:...|
| sensor_4|         22|      59|2026-04-08 10:44:...|
| sensor_0|         37|     

In [0]:
spark.sql("SELECT * FROM silver_iot").show() #silver testing

+---------+-----------+--------+--------------------+
|device_id|temperature|humidity|           timestamp|
+---------+-----------+--------+--------------------+
| sensor_0|         21|      56|2026-04-08 10:44:...|
| sensor_0|         22|      55|2026-04-08 10:44:...|
| sensor_0|         44|      32|2026-04-08 10:44:...|
| sensor_4|         40|      74|2026-04-08 10:44:...|
| sensor_0|         27|      32|2026-04-08 10:44:...|
| sensor_0|         41|      53|2026-04-08 10:44:...|
| sensor_3|         22|      72|2026-04-08 10:44:...|
| sensor_3|         43|      33|2026-04-08 10:44:...|
| sensor_0|         39|      83|2026-04-08 10:44:...|
| sensor_0|         39|      78|2026-04-08 10:44:...|
| sensor_4|         27|      74|2026-04-08 10:44:...|
| sensor_0|         43|      56|2026-04-08 10:44:...|
| sensor_3|         35|      81|2026-04-08 10:44:...|
| sensor_0|         25|      80|2026-04-08 10:44:...|
| sensor_0|         32|      60|2026-04-08 10:44:...|
| sensor_0|         21|     

In [0]:
spark.sql("SELECT * FROM gold_avg").show() #gold testing

+---------+---------------+
|device_id|avg_temperature|
+---------+---------------+
| sensor_4|          33.33|
| sensor_0|           32.3|
| sensor_2|          32.97|
| sensor_3|         32.225|
| sensor_1|         32.225|
+---------+---------------+



In [0]:
spark.sql("SELECT * FROM gold_alerts").show() #gold alerts

+---------+-----------+--------+--------------------+
|device_id|temperature|humidity|           timestamp|
+---------+-----------+--------+--------------------+
| sensor_0|         44|      32|2026-04-08 10:44:...|
| sensor_0|         41|      53|2026-04-08 10:44:...|
| sensor_3|         43|      33|2026-04-08 10:44:...|
| sensor_0|         43|      56|2026-04-08 10:44:...|
| sensor_3|         42|      90|2026-04-08 10:44:...|
| sensor_2|         42|      67|2026-04-08 10:44:...|
| sensor_2|         41|      83|2026-04-08 10:44:...|
| sensor_2|         42|      87|2026-04-08 10:44:...|
| sensor_3|         45|      51|2026-04-08 10:44:...|
| sensor_2|         45|      59|2026-04-08 10:44:...|
| sensor_2|         45|      62|2026-04-08 10:44:...|
| sensor_2|         44|      55|2026-04-08 10:44:...|
| sensor_3|         44|      56|2026-04-08 10:44:...|
| sensor_4|         45|      69|2026-04-08 10:44:...|
| sensor_2|         41|      54|2026-04-08 10:44:...|
| sensor_4|         44|     

In [0]:
#bar chart Visualization 
df = spark.sql("SELECT * FROM gold_avg")
display(df)

device_id,avg_temperature
sensor_4,32.61
sensor_0,32.875
sensor_2,33.42
sensor_3,31.81
sensor_1,32.61


Databricks visualization. Run in Databricks to view.

In [0]:
spark.sql("SHOW TABLES").show()

+--------+------------+-----------+
|database|   tableName|isTemporary|
+--------+------------+-----------+
| default|  bronze_iot|      false|
| default| gold_alerts|      false|
| default|gold_anomaly|      false|
| default|    gold_avg|      false|
| default|  silver_iot|      false|
+--------+------------+-----------+

